---
#### 6-2.1 항공권 가격예측 모델 성능 개선 
- https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction
---

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
import warnings
warnings.filterwarnings("ignore")

In [2]:
# data loading 
df = pd.read_csv('data/Clean_Dataset.csv')
print(df.shape)
df.head()

(300153, 12)


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [3]:
# Unnamed: 0 컬럼 삭제 
df = df.drop('Unnamed: 0', axis=1)
# flight(편명) 삭제 ~ 개수가 너무 많고, 항공사+출발지+도착지 정보로 대체 가능 판단 
df = df.drop('flight', axis=1)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   airline           300153 non-null  object 
 1   source_city       300153 non-null  object 
 2   departure_time    300153 non-null  object 
 3   stops             300153 non-null  object 
 4   arrival_time      300153 non-null  object 
 5   destination_city  300153 non-null  object 
 6   class             300153 non-null  object 
 7   duration          300153 non-null  float64
 8   days_left         300153 non-null  int64  
 9   price             300153 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 22.9+ MB


In [5]:
df.isnull().sum()

airline             0
source_city         0
departure_time      0
stops               0
arrival_time        0
destination_city    0
class               0
duration            0
days_left           0
price               0
dtype: int64

In [6]:
# 범주형 원핫 인코딩 
cat_cols = df.select_dtypes(include='object').columns
print(cat_cols)

df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df.head()

Index(['airline', 'source_city', 'departure_time', 'stops', 'arrival_time',
       'destination_city', 'class'],
      dtype='object')


,duration,days_left,price,airline_Air_India,airline_GO_FIRST,airline_Indigo,airline_SpiceJet,airline_Vistara,source_city_Chennai,source_city_Delhi,...,arrival_time_Evening,arrival_time_Late_Night,arrival_time_Morning,arrival_time_Night,destination_city_Chennai,destination_city_Delhi,destination_city_Hyderabad,destination_city_Kolkata,destination_city_Mumbai,class_Economy
0,2.17,1,5953,0,0,0,1,0,0,1,...,0,0,0,1,0,0,0,0,1,1
1,2.33,1,5953,0,0,0,1,0,0,1,...,0,0,1,0,0,0,0,0,1,1
2,2.17,1,5956,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,1
3,2.25,1,5955,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,1
4,2.33,1,5955,0,0,0,0,1,0,1,...,0,0,1,0,0,0,0,0,1,1


In [7]:
# 타겟 정의 및 검증데이터 분리 
y = df['price']
X = df.drop('price', axis=1)

from sklearn.model_selection import train_test_split 
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42) 

In [8]:
# models 
from sklearn.ensemble import RandomForestRegressor 
from lightgbm import LGBMRegressor 
from xgboost import XGBRegressor 

rf = RandomForestRegressor(random_state=120)
lgb = LGBMRegressor(random_state=120, n_jobs=-1)
xgb = XGBRegressor(random_state=120, n_jobs=-1)

rf.fit(X_train, y_train)
lgb.fit(X_train, y_train)
xgb.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001148 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 361
[LightGBM] [Info] Number of data points in the train set: 210107, number of used features: 30
[LightGBM] [Info] Start training from score 20895.951582


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [9]:
# eval 
from sklearn.metrics import root_mean_squared_error, r2_score 

rf_pred = rf.predict(X_valid)
lgb_pred = lgb.predict(X_valid)
xgb_pred = xgb.predict(X_valid)

rf_rmse = root_mean_squared_error(y_valid, rf_pred)
lgb_rmse = root_mean_squared_error(y_valid, lgb_pred)
xgb_rmse = root_mean_squared_error(y_valid, xgb_pred)

rf_r2 = r2_score(y_valid, rf_pred)
lgb_r2 = r2_score(y_valid, lgb_pred)
xgb_r2 = r2_score(y_valid, xgb_pred)

print("=== RMSE, R2 score ====")
result_df = pd.DataFrame(
    [
        [rf_rmse, lgb_rmse, xgb_rmse],
        [rf_r2,   lgb_r2,   xgb_r2]
    ],
    index=['RMSE', 'R2 score'],
    columns=['Random Forest', 'LightGBM', 'XGBoost']
)

result_df

=== RMSE, R2 score ====


,Random Forest,LightGBM,XGBoost
RMSE,2810.658664,3947.197666,3526.927734
R2 score,0.984651,0.969727,0.975830


In [14]:
######################################################################
# 모델튜닝 효과가 높은 것으로 알려진 XGBoost에 Random Search를 적용하여 최적화 
######################################################################
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from xgboost import callback

# 1차 탐색 
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=120,
    n_jobs=-1
)

param_dist = {
    'n_estimators': randint(500, 1500),
    'learning_rate': uniform(0.01, 0.2),
    'max_depth': randint(3, 10),
    'min_child_weight': randint(1, 10),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5)
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=40,
    scoring='r2',
    cv=5,
    random_state=120,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best Params:", random_search.best_params_)
print("Best CV R2:", random_search.best_score_)

# 1차 탐색결과 기반 최적화 
best_params = random_search.best_params_

final_xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=120,
    n_jobs=-1,
    n_estimators=3000,          # 충분히 크게
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    min_child_weight=best_params['min_child_weight'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    gamma=best_params['gamma'], 
    early_stopping_rounds=100
)

final_xgb.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# 최종결과 
y_pred = final_xgb.predict(X_valid)

rmse = root_mean_squared_error(y_valid, y_pred)
r2 = r2_score(y_valid, y_pred)

print("Final RMSE:", rmse)
print("Final R2:", r2)
print("Best Iteration:", final_xgb.best_iteration)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Params: {'colsample_bytree': np.float64(0.9763663307535211), 'gamma': np.float64(3.0580605144682043), 'learning_rate': np.float64(0.09003781866761863), 'max_depth': 9, 'min_child_weight': 7, 'n_estimators': 1478, 'subsample': np.float64(0.9873152798487693)}
Best CV R2: 0.9876712083816528
Final RMSE: 2439.92724609375
Final R2: 0.9884328246116638
Best Iteration: 1596
